<a href="https://colab.research.google.com/github/demsaid400-cpu/DI-BOOTCAMP/blob/main/Week8/Day1/Daily_Challenge/Daily_Challenge_LangChain_OpenSource_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Daily Challenge: LangChain Pipelines with Open-Source LLMs (Student)
Use this guided notebook with TODOs. Runs on CPU with small HF models (e.g., flan-t5-small).

## What you'll learn
- Set up LangChain with lightweight open-source models.
- Build an LLMChain using a prompt template.
- Compose a two-step Runnable pipeline (summary ? bullets).
- Bonus: add a simple conversation chain with memory.

## What you will create
- Installed environment for LangChain + transformers.
- LLMChain that rewrites text in a simpler style.
- Runnable pipeline that summarizes then bullet-izes text.
- (Bonus) Conversation chain showing memory.

## Part 1: Environment setup (fast)
Install needed packages. CPU is fine for tiny models.

In [1]:
# TODO: verify hardware (optional)
!nvidia-smi || echo "CPU runtime"

/bin/bash: line 1: nvidia-smi: command not found
CPU runtime


In [12]:
# TODO: install dependencies
# We pin numpy<2.0.0 to resolve binary incompatibility in the current Colab environment
!pip install "numpy>=1.26.4,<2.0.0" "transformers>=4.37.2" "langchain>=0.1.7" "langchain-community" "langchain-core" accelerate -q

## Part 2: Load a tiny model and build your first LLMChain
Use a small model (e.g., google/flan-t5-small) to keep inference quick.

In [3]:
# TODO: import libs
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from langchain_community.llms import HuggingFacePipeline
from langchain import PromptTemplate, LLMChain

In [4]:
# TODO: choose a small model
model_name = "google/flan-t5-small"  # small footprint for CPU

In [5]:
# TODO: load tokenizer and model
model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [6]:
# TODO: create a generation pipeline
gen_pipeline = pipeline(
    task="text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,
)
llm = HuggingFacePipeline(pipeline=gen_pipeline)

In [7]:
# TODO: build prompt + LLMChain for friendly rewriting
template = "Rewrite this text to be simpler for beginners: {text}"
prompt = PromptTemplate(template=template, input_variables=["text"])
chain = LLMChain(prompt=prompt, llm=llm)

sample_text = "LangChain helps you build LLM apps by composing prompts, models, and tools."
rewritten = chain.run(text=sample_text)
print(rewritten)

/usr/local/lib/python3.12/dist-packages/langchain_core/_api/deprecation.py:117: LangChainDeprecationWarning: The function `run` was deprecated in LangChain 0.1.0 and will be removed in 0.2.0. Use invoke instead.
  warn_deprecated(


LangChain helps you build LLM apps by composing prompts, models, and tools.


## Part 3: Two-step pipeline (summary ? bullets)
Summarize a paragraph, then turn it into 3 bullets using the same LLM.

In [8]:
from langchain.schema.runnable import RunnableLambda
from langchain import PromptTemplate

# To-Do: define run templates
summary_prompt = PromptTemplate(
    template="Summarize this text in one short sentence: {paragraph}",
    input_variables=["paragraph"],
)

bullets_prompt = PromptTemplate(
    template="Convert this summary into exactly 3 bullet points: {summary}",
    input_variables=["summary"],
)

In [9]:
# First stage: paragraph -> summary (string)
summary_chain = summary_prompt | llm

# Full chain:
# 1. Take input {"paragraph": ...}
# 2. Run summary_chain to get a summary string
# 3. Wrap into {"summary": summary}
# 4. Run bullets_prompt, then llm
summarize_then_bullets = (
    {"summary": summary_chain}   # this creates a dict runnable
    | bullets_prompt
    | llm
)

In [10]:
paragraph = """LangChain is a framework for building applications with large language models by composing prompts, models, and tools. It supports chains, agents, and retrieval workflows."""
bullets_output = summarize_then_bullets.invoke({"paragraph": paragraph})
print(bullets_output)


The LangChain framework is a framework for building applications with large language models by composing prompts, models, and tools.


## Part 4 (Bonus): Conversation chain with memory
Show how two turns keep context.

In [11]:
# TODO: build a simple conversation chain
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory()
convo = ConversationChain(llm=llm, memory=memory, verbose=False)

reply1 = convo.predict(input="Hi there! What's LangChain?")
reply2 = convo.predict(input="Can it help me build a simple chatbot?")
print("Turn 1:", reply1)
print("Turn 2:", reply2)

Turn 1: LangChain is a fictional character in the "LangChain" series.
Turn 2: It's a simple chatbot.


```markdown
## Our observations
- **Latency**: The latency is relatively low since we are using `flan-t5-small`, which is optimized for speed even on CPU runtimes.
- **Quality**: The quality is basic. While it follows the prompt structure, the small parameter count means it may struggle with complex reasoning compared to larger models.
- **Quirks**: You might notice very short answers or repetitive phrasing. Occasionally, the '3 bullet points' instruction might be ignored if the input text is too simple for the model to decompose.
```